<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/10-graph-neural-networks-geometric-deep-learning.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **图神经网络与几何深度学习** {#graph-neural-networks-geometric-deep-learning}

图像、音频和普通序列位于规则网格上：每个像素或 token 都有可预测的坐标，也有固定的“下一个位置”概念。图则用实体集合与关系集合 $G=(V,E)$ 表示对象。不同节点的邻居数量不同，节点标识符本身是任意的，而且改变节点在存储中的顺序不应改变对象的含义。**几何深度学习（geometric deep learning）**研究计算过程如何尊重此类对称性与数据域；图神经网络（GNN）是其中使用最广泛的离散形式。

本章从头到尾使用同一个真实网络：[NetworkX 中的 Zachary Karate Club graph](https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html)。Wayne Zachary 记录了 34 名俱乐部成员之间的互动，随后观察到俱乐部围绕两位领导者分裂。NetworkX 提供 78 条无向边、表示互动次数的边权重，以及节点属性 `club` 中记录的实际派系。NetworkX 实现采用 BSD-3-Clause 许可证；其文档引用了 [1977 年的原始研究](https://www.jstor.org/stable/3629752)，但没有为数据单独声明许可证。因此，本章只使用 NetworkX 随库提供的小型拓扑与元数据，并明确标注出处。

固定实验是**传导式节点分类（transductive node classification）**：8 个有标签节点用于训练，8 个节点用于选择 checkpoint，18 个节点留作最终机制检查，同时模型始终可以看到整个图的拓扑。后面还会建立干净的边划分来演示链接预测。这个网络规模很小，具有特定历史背景，而且节点之间存在社会相关性，不能用于证明 GNN 的普遍优势。它的价值在于：每个矩阵、邻域、注意力系数与泄漏边界都可以直接检查。

![Zachary Karate Club 网络，节点颜色表示分裂后观察到的两个派系。](assets/dl10-karate-club.svg){fig-align="center" width="76%" fig-alt="包含 34 个编号节点、两种派系颜色以及加权互动边的 Zachary Karate Club 图。"}

*本章根据 [NetworkX Karate Club graph](https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html) 生成的原创数据可视化；拓扑与元数据可追溯至 Zachary（1977）。*

### **为什么图结构数据不同** {#why-graph-structured-data-is-different}

当关系本身属于输入，而不是偶然出现的相关性时，图表示最有价值。在分子中，原子是节点、化学键是边；在推荐系统中，用户和物品构成二部互动图；在道路网络中，路口通过带方向和权重的道路连接。把这些对象直接展平为向量会丢失哪些特征属于相邻实体；强行放进任意稠密网格则会虚构原本不存在的空间邻居。

模型设计由三个性质决定。第一，**不规则邻域**要求对大小可变的集合做聚合。第二，**置换对称性**要求节点输出随节点顺序一起置换，而图级输出保持不变。第三，**关系依赖性**意味着样本不会自动相互独立：随机节点或边划分可能通过消息传递泄漏标签、未来事件、重复实体或留出的边。

GNN 通过在所有节点上共享局部更新规则，并采用与顺序无关的聚合来处理前两个性质。它不会自动解决第三个性质。评估协议、特征来源、图构造方式和部署假设与层公式同等重要。

<details>
<summary><strong>PyTorch 与 NetworkX：建立整章共享的 Karate Club 实验</strong></summary>

```python
import copy
import random

import networkx as nx
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)


def seed_everything(seed=1010):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
graph = nx.karate_club_graph()
nodes = sorted(graph.nodes())
num_nodes = graph.number_of_nodes()

# Binary topology drives message passing; the original interaction counts are
# retained separately as edge attributes for a later relation-aware example.
adjacency = torch.tensor(
    nx.to_numpy_array(graph, nodelist=nodes, weight=None), dtype=torch.float32
)
weighted_adjacency = torch.tensor(
    nx.to_numpy_array(graph, nodelist=nodes, weight="weight"), dtype=torch.float32
)
labels = torch.tensor(
    [0 if graph.nodes[node]["club"] == "Mr. Hi" else 1 for node in nodes],
    dtype=torch.long,
)

undirected_edges = torch.tensor(list(graph.edges()), dtype=torch.long).T
edge_index = torch.cat([undirected_edges, undirected_edges.flip(0)], dim=1)
degree = adjacency.sum(1)
clustering = torch.tensor(
    [nx.clustering(graph, node) for node in nodes], dtype=torch.float32
)
node_features = torch.stack(
    [degree / degree.max(), clustering, torch.ones(num_nodes)], dim=1
)
identity_features = torch.eye(num_nodes)

# One deterministic, class-stratified transductive split is reused throughout.
rng = np.random.default_rng(1010)
train_nodes, val_nodes, test_nodes = [], [], []
for class_id in (0, 1):
    members = np.flatnonzero(labels.numpy() == class_id)
    rng.shuffle(members)
    train_nodes.extend(members[:4])
    val_nodes.extend(members[4:8])
    test_nodes.extend(members[8:])


def mask_from(indices):
    mask = torch.zeros(num_nodes, dtype=torch.bool)
    mask[torch.tensor(indices, dtype=torch.long)] = True
    return mask


train_mask = mask_from(train_nodes)
val_mask = mask_from(val_nodes)
test_mask = mask_from(test_nodes)

assert num_nodes == 34 and graph.number_of_edges() == 78
assert torch.equal(adjacency, adjacency.T) and torch.all(adjacency.diag() == 0)
assert train_mask.sum() == 8 and val_mask.sum() == 8 and test_mask.sum() == 18
assert not (train_mask & val_mask).any() and not (train_mask & test_mask).any()
print({
    "NetworkX": nx.__version__,
    "nodes": num_nodes,
    "undirected edges": graph.number_of_edges(),
    "split": (int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum())),
    "feature shape": tuple(node_features.shape),
})
```

</details>

三个结构特征分别是归一化度数、局部聚类系数和常数通道。代码还保留 identity features，因为经典半监督 GCN 演示通常为每个节点学习独立的基向量方向。这种表示明确属于传导式学习：新增节点没有训练过的 identity 列。后面的 GraphSAGE 示例改用结构特征，以揭示“学习节点 ID”与“学习可复用邻域函数”之间的差别。


### **图表示与学习任务** {#graph-representations-learning-tasks}

同一个图可以存储为 edge list、adjacency matrix、sparse coordinate list 或 neighborhood dictionary。对于 $N=|V|$ 个节点，稠密邻接矩阵 $A\in\{0,1\}^{N\times N}$ 在 $i$ 与 $j$ 相连时令 $A_{ij}=1$。即使图只有 $E\ll N^2$ 条边，它仍需要 $O(N^2)$ 内存。Edge index 只存端点对，成本为 $O(E)$；无向边通常会展开为两个方向的消息，因此本章的 78 条关系变成 156 列。

节点特征组成 $X\in\mathbb{R}^{N\times F}$，边特征组成 $E_f\in\mathbb{R}^{|E|\times F_e}$，还可以加入描述整个对象的图级特征。用置换矩阵 $P$ 重新编号后，$X'=PX$ 且 $A'=PAP^\top$。节点模型应满足

$$
f(PX,PAP^\top)=P f(X,A),
$$

这称为**置换等变性（permutation equivariance）**。图级读出 $r$ 应满足 $r(PX,PAP^\top)=r(X,A)$，即**置换不变性（permutation invariance）**。它们是架构必须满足的契约，而不是可选的数据增强偏好。

预测目标决定输出结构。**节点级**任务对成员、论文或原子分类；**边级**任务预测链接、关系类型或化学键属性；**图级**任务把整个分子、程序图或场景映射到一个标签或标量。第四类是**图生成**，其中有效性约束和置换等变概率还需要额外机制。

![节点、边和图级目标需要不同的读出操作。](assets/dl10-readout-levels.svg){fig-align="center" width="72%" fig-alt="对比节点分类、边打分和图级不变池化的三个面板。"}

*原创教学图，任务分类依据 [Distill 的 GNN 入门文章](https://distill.pub/2021/gnn-intro/)；该文章的文字和图示采用 CC BY 4.0。*

<details>
<summary><strong>Python：比较 edge list、adjacency 与 sparse edge index 视图</strong></summary>

```python
edge_list = sorted(tuple(sorted(edge)) for edge in graph.edges())
neighbors_of_zero = sorted(graph.neighbors(0))

# Relabeling a graph is a simultaneous permutation of rows and columns.
permutation = torch.randperm(num_nodes, generator=torch.Generator().manual_seed(1011))
permuted_adjacency = adjacency[permutation][:, permutation]
permuted_features = node_features[permutation]

assert len(edge_list) == 78
assert edge_index.shape == (2, 156)  # both directions are explicit
assert torch.equal(permuted_adjacency.sum(1), degree[permutation])
assert torch.equal(permuted_features[:, -1], torch.ones(num_nodes))
print({
    "node 0 neighbors": neighbors_of_zero,
    "adjacency shape": tuple(adjacency.shape),
    "edge-index shape": tuple(edge_index.shape),
    "density": float(adjacency.sum() / num_nodes**2),
})
```

</details>

对于 34 节点的教学图，稠密矩阵能让代数关系清晰可见。PyTorch Geometric 和 DGL 等生产库采用稀疏消息传递 kernel，使单层复杂度更接近 $O(EF)$，而不是 $O(N^2F)$。稀疏存储并不会消除高度节点的负载不均、重复边语义或昂贵的邻域扩张；这些仍然是系统层面的难题。


### **消息传递框架** {#message-passing-framework}

多数实用 GNN 层都可以拆成**消息（message）**、**聚合（aggregate）**和**更新（update）**三个操作。对第 $k$ 层节点 $i$，

$$
m_{ji}^{(k)}=\phi^{(k)}\!\left(h_i^{(k-1)},h_j^{(k-1)},e_{ji}\right),\qquad
m_i^{(k)}=\bigoplus_{j\in\mathcal{N}(i)}m_{ji}^{(k)},\qquad
h_i^{(k)}=\gamma^{(k)}\!\left(h_i^{(k-1)},m_i^{(k)}\right).
$$

$h_i^{(k)}$ 是节点 $i$ 的表示，$e_{ji}$ 是可选边特征，$\phi$ 构造有方向的消息，$\bigoplus$ 是 sum、mean 或 max 等与顺序无关的算子，$\gamma$ 把聚合后的邻域信息与旧状态组合起来。这正是 [PyTorch Geometric `MessagePassing` API](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_gnn.html) 所形式化的抽象。

![消息传递层依次执行共享消息函数、顺序不变聚合和节点更新。](assets/dl10-message-passing.svg){fig-align="center" width="74%" fig-alt="三个邻居节点经过 message、aggregate 和 update 阶段完成消息传递。"}

*原创教学图，依据 [PyTorch Geometric 文档](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_gnn.html)中的消息传递公式以及 [Distill GNN 入门文章](https://distill.pub/2021/gnn-intro/)重新组织。*

经过一层后，$h_i$ 依赖一跳邻居；经过 $K$ 层后，它可能依赖 $K$ 跳范围内的节点。这个**感受野（receptive field）**结论描述的是可能存在的信息路径，并不保证模型真的保留并利用这些信息。聚合可能丢失重数，归一化可能衰减信号，狭窄割边还可能压缩远距离证据。

<details>
<summary><strong>PyTorch：实现一个满足置换等变性的消息传递层</strong></summary>

```python
def mean_aggregate(features, binary_adjacency, include_self=True):
    mixing = binary_adjacency.clone()
    if include_self:
        mixing = mixing + torch.eye(mixing.shape[0])
    return mixing @ features / mixing.sum(1, keepdim=True).clamp_min(1.0)


class MessagePassingLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.update = nn.Linear(2 * input_dim, output_dim)

    def forward(self, features, binary_adjacency):
        neighbor_message = mean_aggregate(features, binary_adjacency)
        return torch.relu(self.update(torch.cat([features, neighbor_message], dim=1)))


seed_everything(1012)
message_layer = MessagePassingLayer(node_features.shape[1], 8)
node_states = message_layer(node_features, adjacency)
permuted_states = message_layer(permuted_features, permuted_adjacency)

# The output follows the node permutation: graph labels may change, semantics do not.
assert node_states.shape == (num_nodes, 8)
assert torch.allclose(permuted_states, node_states[permutation], atol=1e-6)
print({"node 0 state": node_states[0].detach().round(decimals=3).tolist(),
       "permutation equivariant": True})
```

</details>

示例使用 mean，并在学习更新前把目标节点自身状态与邻居均值拼接。Mean 能控制不同度数下的尺度，却无法区分平均值相同的邻域。Sum 保留重数，更适合计数，但幅度随邻域规模增加。Max 突出最强坐标，却丢弃频率。因此，聚合方式编码的是归纳偏置，并不存在普遍最优的默认选项。


### **图卷积网络** {#graph-convolutional-networks}

[Kipf 与 Welling](https://arxiv.org/abs/1609.02907)提出的 Graph Convolutional Network（GCN）使用固定的度归一化邻域平均，再进行可学习的通道变换。加入自环 $\hat A=A+I$，定义 $\hat D_{ii}=\sum_j\hat A_{ij}$，传播公式为

$$
H^{(k+1)}=\sigma\!\left(\hat D^{-1/2}\hat A\hat D^{-1/2}H^{(k)}W^{(k)}\right).
$$

$H^{(k)}\in\mathbb{R}^{N\times F_k}$ 包含节点状态，$W^{(k)}\in\mathbb{R}^{F_k\times F_{k+1}}$ 混合特征通道，对称归一化因子把每条边的贡献除以 $\sqrt{\hat d_i\hat d_j}$。这样，高度节点既不会累积失控的总和，也不会完全支配每个低度邻居。原论文从谱图卷积的局部一阶近似推导该层，但也可以直接把它理解为归一化消息传递。

两层让每个节点使用两跳证据。在传导式 Karate 实验中，loss 只在 `train_mask` 上计算，但无标签节点仍参与传播。只有当部署时同一个完整图可用，这个协议才成立。如果测试节点或它们的边在训练时本应不可见，那么即使标签被 mask，这个协议仍然错误。

![GCN、GraphSAGE 与 GAT 的核心差别是如何选择和加权邻域消息。](assets/dl10-gnn-operators.svg){fig-align="center" width="78%" fig-alt="对比度归一化 GCN、采样均值 GraphSAGE 和注意力加权 GAT 聚合的三面板图。"}

*原创对比图，依据 [GCN](https://arxiv.org/abs/1609.02907)、[GraphSAGE](https://arxiv.org/abs/1706.02216) 与 [GAT](https://arxiv.org/abs/1710.10903) 原始论文。*

<details>
<summary><strong>PyTorch：用矩阵运算训练两层 GCN</strong></summary>

```python
def gcn_normalize(binary_adjacency):
    adjacency_with_self = binary_adjacency + torch.eye(binary_adjacency.shape[0])
    inverse_sqrt_degree = adjacency_with_self.sum(1).pow(-0.5)
    return (
        inverse_sqrt_degree[:, None]
        * adjacency_with_self
        * inverse_sqrt_degree[None, :]
    )


class KarateGCN(nn.Module):
    def __init__(self, propagation, input_dim, hidden_dim=16):
        super().__init__()
        self.propagation = propagation
        self.input_layer = nn.Linear(input_dim, hidden_dim, bias=False)
        self.output_layer = nn.Linear(hidden_dim, 2, bias=False)

    def forward(self, features):
        hidden = torch.relu(self.propagation @ self.input_layer(features))
        logits = self.propagation @ self.output_layer(hidden)
        return logits, hidden


def fit_node_model(model, features, epochs=300, learning_rate=0.02):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=5e-4)
    best_val_loss = float("inf")
    best_state = None
    for _ in range(epochs):
        model.train()
        logits, _ = model(features)
        loss = F.cross_entropy(logits[train_mask], labels[train_mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logits, _ = model(features)
            val_loss = F.cross_entropy(val_logits[val_mask], labels[val_mask]).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits, hidden = model(features)
    predictions = logits.argmax(1)
    return {
        "validation loss": best_val_loss,
        "test macro F1": f1_score(labels[test_mask], predictions[test_mask], average="macro"),
    }, logits, hidden


seed_everything(1013)
normalized_adjacency = gcn_normalize(adjacency)
gcn_model = KarateGCN(normalized_adjacency, input_dim=num_nodes)
gcn_metrics, gcn_logits, gcn_embeddings = fit_node_model(gcn_model, identity_features)

assert torch.allclose(normalized_adjacency, normalized_adjacency.T)
assert gcn_logits.shape == (num_nodes, 2) and gcn_embeddings.shape == (num_nodes, 16)
print({key: round(float(value), 4) for key, value in gcn_metrics.items()})
```

</details>

Identity input 为每个已观察成员提供可学习的基向量方向，使拓扑本身足以完成机制演示。但不修改输入层时，它无法嵌入成员 34。报告的 macro F1 只来自一个很小的固定划分，应与 validation loss 和简单 baseline 一起阅读，不能作为 GCN 普遍最优的证据。在大型稀疏图上，这里展示的稠密乘法必须替换为稀疏 gather/scatter 操作。


### **GraphSAGE** {#graphsage}

[GraphSAGE](https://arxiv.org/abs/1706.02216) 把问题从“节点 $i$ 对应哪个 embedding 参数”改成“哪个函数能把节点特征与采样邻域映射为 embedding”。Mean aggregator 可以写成

$$
\bar h_{\mathcal N(i)}^{(k)}=\frac{1}{|S_k(i)|}\sum_{j\in S_k(i)}h_j^{(k-1)},\qquad
h_i^{(k)}=\sigma\!\left(W^{(k)}[h_i^{(k-1)}\Vert \bar h_{\mathcal N(i)}^{(k)}]\right),
$$

其中 $S_k(i)\subseteq\mathcal N(i)$ 是采样邻居集合，$\Vert$ 表示拼接。参数属于聚合函数，而不是节点查找表；因此，只要新节点有特征和邻域，同一个函数就能处理它。这是一个**归纳式接口**，并不自动保证模型能准确处理分布已经变化的新节点。

采样把 $K$ 层计算控制在每个目标节点约 $\prod_k s_k$ 条采样路径内，避免展开全部邻居。它降低成本并支持 mini-batch，却会引入估计方差。高度邻域可能代表不足，稀有关系类型可能完全消失，朴素递归采样还会为相互重叠的感受野重复计算。

<details>
<summary><strong>PyTorch：训练 mean GraphSAGE 层并检查采样误差</strong></summary>

```python
class MeanSAGELayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(2 * input_dim, output_dim)

    def forward(self, features, binary_adjacency):
        neighbor_mean = mean_aggregate(features, binary_adjacency, include_self=False)
        return torch.relu(self.linear(torch.cat([features, neighbor_mean], dim=1)))


class KarateGraphSAGE(nn.Module):
    def __init__(self, binary_adjacency, input_dim, hidden_dim=16):
        super().__init__()
        self.binary_adjacency = binary_adjacency
        self.sage = MeanSAGELayer(input_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, features):
        hidden = self.sage(features, self.binary_adjacency)
        return self.classifier(hidden), hidden


def sampled_adjacency(graph_object, fan_out, seed):
    generator = random.Random(seed)
    sampled = torch.zeros((num_nodes, num_nodes))
    for target in nodes:
        candidates = list(graph_object.neighbors(target))
        chosen = generator.sample(candidates, min(fan_out, len(candidates)))
        sampled[target, chosen] = 1.0
    return sampled


seed_everything(1014)
sage_model = KarateGraphSAGE(adjacency, input_dim=node_features.shape[1])
sage_metrics, sage_logits, sage_embeddings = fit_node_model(
    sage_model, node_features, learning_rate=0.015
)
sampled_graph = sampled_adjacency(graph, fan_out=2, seed=1014)
full_neighbor_mean = mean_aggregate(node_features, adjacency, include_self=False)
sampled_neighbor_mean = mean_aggregate(node_features, sampled_graph, include_self=False)

assert sampled_graph.sum(1).max() <= 2
assert sage_embeddings.shape == (num_nodes, 16)
print({
    "sampled directed messages": int(sampled_graph.sum()),
    "node 0 mean approximation error": round(
        float((full_neighbor_mean[0] - sampled_neighbor_mean[0]).norm()), 4
    ),
    "test macro F1": round(float(sage_metrics["test macro F1"]), 4),
})
```

</details>

该模型接收 degree、clustering 和常数，而不是节点 identity。这些特征可以在另一个图上计算，不过取值范围和语义可能发生变化。两邻居 sampler 刻意制造可观察的近似误差；真实系统需要逐层选择 fan-out，按关系或重要性采样，缓存邻域，并报告多个采样 seed 之间的方差。


### **图注意力网络** {#graph-attention-networks}

GCN 根据度数固定邻居权重。[Graph Attention Network（GAT）](https://arxiv.org/abs/1710.10903)则从节点内容中学习权重。对变换后的状态 $z_i=Wh_i$，原始单头层计算

$$
e_{ij}=\operatorname{LeakyReLU}\!\left(a^\top[z_i\Vert z_j]\right),\qquad
\alpha_{ij}=\frac{\exp(e_{ij})}{\sum_{k\in\mathcal N(i)\cup\{i\}}\exp(e_{ik})},\qquad
h_i'=\sigma\!\left(\sum_j\alpha_{ij}z_j\right).
$$

Softmax 是局部的：系数在**每个目标节点的入邻域内部**和为 1，而不是在整张图上归一化。Multi-head GAT 使用独立投影重复计算。中间层通常拼接多个 head 以增加容量，最终层也可以对 heads 取平均来稳定预测。

Attention 能压低无用邻居，并针对不同样本调整权重，但并非没有成本。它需要逐边计算分数、存储系数，而且信息仍只能沿输入提供的边流动。原始 additive scoring 形式本身也有表达限制；若没有干预实验，学习到的系数不能直接当作因果解释。

<details>
<summary><strong>PyTorch：实现局部 edge softmax 并训练单头 GAT</strong></summary>

```python
class SingleHeadGATLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.projection = nn.Linear(input_dim, output_dim, bias=False)
        self.attention_source = nn.Parameter(torch.empty(output_dim))
        self.attention_target = nn.Parameter(torch.empty(output_dim))
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.normal_(self.attention_source, std=0.2)
        nn.init.normal_(self.attention_target, std=0.2)

    def forward(self, features, directed_edges):
        self_nodes = torch.arange(features.shape[0])
        edges = torch.cat([directed_edges, torch.stack([self_nodes, self_nodes])], dim=1)
        source, target = edges
        transformed = self.projection(features)
        scores = F.leaky_relu(
            (transformed[source] * self.attention_source).sum(1)
            + (transformed[target] * self.attention_target).sum(1),
            negative_slope=0.2,
        )
        coefficients = torch.zeros_like(scores)
        for target_node in range(features.shape[0]):
            incoming = target == target_node
            coefficients[incoming] = torch.softmax(scores[incoming], dim=0)
        output = torch.zeros_like(transformed)
        output.index_add_(0, target, coefficients[:, None] * transformed[source])
        return F.elu(output), edges, coefficients


class KarateGAT(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()
        self.gat = SingleHeadGATLayer(input_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)
        self.last_edges = None
        self.last_attention = None

    def forward(self, features):
        hidden, edges, coefficients = self.gat(features, edge_index)
        self.last_edges = edges
        self.last_attention = coefficients.detach()
        return self.classifier(hidden), hidden


seed_everything(1015)
gat_model = KarateGAT(node_features.shape[1])
gat_metrics, gat_logits, gat_embeddings = fit_node_model(
    gat_model, node_features, learning_rate=0.012
)
with torch.no_grad():
    gat_model(node_features)
target_indices = gat_model.last_edges[1]
attention_sums = torch.zeros(num_nodes)
attention_sums.index_add_(0, target_indices, gat_model.last_attention)

assert torch.allclose(attention_sums, torch.ones(num_nodes), atol=1e-5)
assert gat_embeddings.shape == (num_nodes, 16)
print({"node 0 incoming coefficients": gat_model.last_attention[target_indices == 0].round(decimals=3).tolist(),
       "test macro F1": round(float(gat_metrics["test macro F1"]), 4)})
```

</details>

显式循环只适合 34 个节点，因为它能让归一化范围完全透明。稀疏库会按 target index 实现 segment softmax。常见错误是对所有边做全局归一化，从而把无关邻域耦合起来；另一个错误是交换 source 和 target，导致代码悄悄对出消息而不是入消息做归一化。


### **节点、边与图级读出** {#node-edge-graph-readout}

消息传递产生节点状态；任务 head 决定这些状态如何变成预测。节点分类器对每个 $h_i$ 应用 MLP 或线性映射。边预测器通过点积、距离、双线性形式、Hadamard product 或 relation-aware MLP 组合端点状态。图预测器把全部节点或边状态汇聚成一个表示。

图级输出的 readout 必须满足置换不变性：

$$
h_G=\operatorname{READOUT}\left(\{h_i:i\in V\}\right).
$$

`sum` 保留图规模信息，也能对重复结构计数；`mean` 比较不同规模图的平均组成；`max` 判断强特征是否存在。Attention pooling 和 virtual global node 是可学习替代方案，但仍需对输入集合进行不变处理。当局部子结构应该组成更高层单元时，可以使用 hierarchical pooling 对图做粗化。

<details>
<summary><strong>PyTorch：从同一 embedding table 得到节点、边和图级输出</strong></summary>

```python
with torch.no_grad():
    node_logits, node_embeddings = gcn_model(identity_features)

unique_source, unique_target = undirected_edges
edge_embeddings = node_embeddings[unique_source] * node_embeddings[unique_target]
edge_scores = edge_embeddings.sum(1)
graph_mean = node_embeddings.mean(0)
graph_sum = node_embeddings.sum(0)
graph_max = node_embeddings.max(0).values

permuted_embeddings = node_embeddings[permutation]
assert node_logits.shape == (34, 2)
assert edge_scores.shape == (78,)
assert torch.allclose(graph_mean, permuted_embeddings.mean(0), atol=1e-6)
assert torch.allclose(graph_sum, graph_mean * num_nodes, atol=1e-6)
print({"node output": tuple(node_logits.shape), "edge output": tuple(edge_scores.shape),
       "graph readouts": [tuple(x.shape) for x in (graph_mean, graph_sum, graph_max)]})
```

</details>

Hadamard 边表示是对称的，因此交换端点不会改变无向边分数。拼接则与顺序有关，除非模型同时处理两个方向。同样，`sum` 和 `mean` 不能随意互换：`sum = mean × number of nodes` 直接说明 mean 删除了哪一种规模信号。


### **异构图与时序图** {#heterogeneous-temporal-graphs}

同构图假设只有一种节点类型和一种边语义。知识图谱、推荐系统和软件依赖图都违反这一假设。异构消息传递层根据关系类型 $r$ 调整计算：

$$
h_i'=\gamma\!\left(h_i,\sum_r\sum_{j\in\mathcal N_r(i)}\phi_r(h_i,h_j,e_{ji})\right).
$$

为每种关系设置独立矩阵最直接，但参数量会随关系数量线性增长。Basis decomposition、参数共享、typed attention 和 metapath 方法在容量、统计效率与内存之间做取舍。不同节点类型还可能拥有不同的原始特征空间，需要先用类型专属 encoder 映射到共享隐藏维度。

时序图为事件加入时间：$(u,v,t,e_{uvt})$。在时间 $t$ 做预测时，可用邻域只能包含 $t'<t$ 的事件。Temporal GNN 可以维护节点 memory、编码时间间隔，或采样最近的历史邻居。如果部署目标是预测未来，随机边划分就是无效的，因为它会让未来互动塑造过去的 embedding。

<details>
<summary><strong>PyTorch：应用关系专属消息，同时不虚构时间戳</strong></summary>

```python
# The original edge weight counts interaction contexts. We bin it only to
# demonstrate relation-specific parameters; this is not a native edge schema.
relation_edges = {"lower_weight": [], "higher_weight": []}
for source, target, attributes in graph.edges(data=True):
    relation = "higher_weight" if attributes["weight"] >= 3 else "lower_weight"
    relation_edges[relation].extend([(source, target), (target, source)])

seed_everything(1016)
relation_weights = {
    "lower_weight": torch.randn(3, 4) / 3**0.5,
    "higher_weight": torch.randn(3, 4) / 3**0.5,
}
relation_output = torch.zeros(num_nodes, 4)
for relation, edges in relation_edges.items():
    directed = torch.tensor(edges, dtype=torch.long).T
    source, target = directed
    messages = node_features[source] @ relation_weights[relation]
    relation_output.index_add_(0, target, messages)

has_timestamps = any("timestamp" in attributes for *_, attributes in graph.edges(data=True))
assert relation_output.shape == (34, 4)
assert sum(len(edges) for edges in relation_edges.values()) == 156
assert not has_timestamps
print({key: len(value) // 2 for key, value in relation_edges.items()},
      "timestamps available:", has_timestamps)
```

</details>

Karate Club 提供互动次数，却没有事件时间戳。代码把权重分为两个**人为构造的**关系，只为演示参数路由；它并未声称 Zachary 记录了两种边类型。更重要的是，示例拒绝从权重模拟时序结论。互动强度回答“在观察到的情境中出现多少次”，时间回答“何时发生”；用前者替代后者会制造虚假的因果顺序。


### **过平滑与过压缩** {#oversmoothing-oversquashing}

深层消息传递有两种不同的失败模式。**过平滑（oversmoothing）**指反复混合邻域后，节点表示越来越相似。GCN propagation 类似 Laplacian smoothing；在连通图上使用足够多层后，有判别力的变化可能坍缩到低维 stationary subspace。[Li、Han 与 Wu](https://arxiv.org/abs/1801.07606)指出，这种 smoothing 既是 GCN 有效的来源，也是其深度限制。

**过压缩（oversquashing）**指迅速扩张的感受野必须把大量远距离信号压进固定宽度状态，或挤过狭窄的图割。[Alon 与 Yahav](https://arxiv.org/abs/2006.05205)说明，即使所有节点表示尚未变得相同，长距离任务也可能已经失败。过平滑关注状态变得不可区分；过压缩关注相关远程信息无法穿过瓶颈。

![过平滑使节点区别坍缩，过压缩则让大量长距离消息挤过狭窄路径。](assets/dl10-failure-modes.svg){fig-align="center" width="76%" fig-alt="对比过平滑造成表示坍缩与过压缩造成狭窄信息瓶颈的双面板图。"}

*原创教学图，依据 [Deeper Insights into GCNs](https://arxiv.org/abs/1801.07606) 与 [On the Bottleneck of GNNs](https://arxiv.org/abs/2006.05205)。*

<details>
<summary><strong>Python：测量 smoothing 并检查拓扑瓶颈 proxy</strong></summary>

```python
# A row-stochastic propagation matrix isolates the smoothing effect.
random_walk = adjacency + torch.eye(num_nodes)
random_walk = random_walk / random_walk.sum(1, keepdim=True)
states = node_features.clone()
diagnostics = []
for layer in range(21):
    class_gap = (states[labels == 0].mean(0) - states[labels == 1].mean(0)).norm()
    diagnostics.append((layer, float(states.var(0).mean()), float(class_gap)))
    states = random_walk @ states

# Topology diagnostics for possible information bottlenecks.
eccentricity = nx.eccentricity(graph)
peripheral_target = max(eccentricity, key=eccentricity.get)
distances = nx.single_source_shortest_path_length(graph, peripheral_target)
shell_sizes = {
    distance: sum(value == distance for value in distances.values())
    for distance in sorted(set(distances.values()))
}
edge_betweenness = nx.edge_betweenness_centrality(graph)
highest_betweenness_edge = max(edge_betweenness, key=edge_betweenness.get)

assert diagnostics[-1][1] < diagnostics[0][1]
assert sum(shell_sizes.values()) == num_nodes
print({
    "variance layer 0 -> 20": (round(diagnostics[0][1], 5), round(diagnostics[-1][1], 5)),
    "class gap layer 0 -> 20": (round(diagnostics[0][2], 4), round(diagnostics[-1][2], 4)),
    "peripheral target and shells": (peripheral_target, shell_sizes),
    "highest edge-betweenness candidate": highest_betweenness_edge,
})
```

</details>

节点特征方差下降是 smoothing diagnostic，不是任务失败的完整证明。Edge betweenness 和感受野 shell size 也只是拓扑 proxy，并不直接测量信息容量。常见缓解手段包括 residual 或 initial-feature connection、normalization、浅层模型、jumping knowledge、graph rewiring、位置/结构编码和 global attention。每种方法针对的原因不同，因此应先诊断，再更改架构。


### **分子建模与推荐系统** {#molecular-modeling-recommendation-systems}

只有先明确图的语义，GNN 抽象才能跨领域迁移。在分子学习中，原子特征包含元素、电荷、芳香性与几何信息；化学键特征包含 bond type，有时还包括距离。节点任务预测原子属性，边任务预测化学键或相互作用，图级 readout 预测分子性质。三维模型还必须满足旋转/平移不变性或等变性；普通二维 GCN 不会因为输入被称作“分子”就自动获得物理对称性。

在推荐系统中，用户和物品属于不同节点类型，观察到的互动是边。模型通常用 dot product 或 MLP 给候选 pair 打分。未观察到的 pair 不一定是负例：用户也许从未看到过该物品。因此 negative sampling 实际上定义了学习问题，而 popularity-biased exposure 会让离线排序指标偏乐观。

Karate graph 可以支持边预测机制检查。为避免结构泄漏，validation 和 test positive edge 会从 encoder 使用的 adjacency 中删除。删除算法保证训练图保持连通，然后从真正的 non-edge 中采样负例。Identity features 使其成为传导式 link model，类似学习 user/item ID；没有 side features 或 inductive encoder 时，它不能给全新实体打分。

<details>
<summary><strong>PyTorch：在留出的 Karate edge 上训练无泄漏链接预测器</strong></summary>

```python
def make_connected_edge_split(graph_object, validation_size=6, test_size=6, seed=1017):
    generator = random.Random(seed)
    candidates = list(graph_object.edges())
    generator.shuffle(candidates)
    training_graph = graph_object.copy()
    held_out = []
    for edge in candidates:
        training_graph.remove_edge(*edge)
        if nx.is_connected(training_graph):
            held_out.append(edge)
        else:
            training_graph.add_edge(*edge, **graph_object.edges[edge])
        if len(held_out) == validation_size + test_size:
            break
    return training_graph, held_out[:validation_size], held_out[validation_size:]


def edge_tensor(edges):
    return torch.tensor(edges, dtype=torch.long).T.contiguous()


def dot_scores(embeddings, edges):
    source, target = edges
    return (embeddings[source] * embeddings[target]).sum(1)


training_graph, validation_positive, test_positive = make_connected_edge_split(graph)
training_positive = list(training_graph.edges())
negative_edges = list(nx.non_edges(graph))
random.Random(1017).shuffle(negative_edges)
training_negative = negative_edges[:len(training_positive)]
validation_negative = negative_edges[len(training_positive):len(training_positive) + len(validation_positive)]
test_negative = negative_edges[len(training_positive) + len(validation_positive):
                               len(training_positive) + len(validation_positive) + len(test_positive)]

training_adjacency = torch.tensor(
    nx.to_numpy_array(training_graph, nodelist=nodes, weight=None), dtype=torch.float32
)


class LinkEncoder(nn.Module):
    def __init__(self, propagation, hidden_dim=16):
        super().__init__()
        self.propagation = propagation
        self.layer1 = nn.Linear(num_nodes, hidden_dim, bias=False)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, features):
        hidden = torch.relu(self.propagation @ self.layer1(features))
        return self.propagation @ self.layer2(hidden)


def binary_edge_loss(embeddings, positive, negative):
    positive_scores = dot_scores(embeddings, edge_tensor(positive))
    negative_scores = dot_scores(embeddings, edge_tensor(negative))
    scores = torch.cat([positive_scores, negative_scores])
    targets = torch.cat([torch.ones_like(positive_scores), torch.zeros_like(negative_scores)])
    return F.binary_cross_entropy_with_logits(scores, targets)


def link_metrics(embeddings, positive, negative):
    scores = torch.cat([
        dot_scores(embeddings, edge_tensor(positive)),
        dot_scores(embeddings, edge_tensor(negative)),
    ]).sigmoid().detach().numpy()
    targets = np.r_[np.ones(len(positive)), np.zeros(len(negative))]
    return roc_auc_score(targets, scores), average_precision_score(targets, scores)


seed_everything(1017)
link_model = LinkEncoder(gcn_normalize(training_adjacency))
optimizer = torch.optim.AdamW(link_model.parameters(), lr=0.02, weight_decay=1e-4)
best_validation_loss, best_link_state = float("inf"), None
for _ in range(300):
    link_model.train()
    embeddings = link_model(identity_features)
    loss = binary_edge_loss(embeddings, training_positive, training_negative)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    link_model.eval()
    with torch.no_grad():
        validation_embeddings = link_model(identity_features)
        validation_loss = binary_edge_loss(
            validation_embeddings, validation_positive, validation_negative
        ).item()
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_link_state = copy.deepcopy(link_model.state_dict())

link_model.load_state_dict(best_link_state)
link_model.eval()
with torch.no_grad():
    link_embeddings = link_model(identity_features)
validation_auc, validation_ap = link_metrics(link_embeddings, validation_positive, validation_negative)
test_auc, test_ap = link_metrics(link_embeddings, test_positive, test_negative)

assert nx.is_connected(training_graph)
assert all(not training_graph.has_edge(*edge) for edge in validation_positive + test_positive)
print({"edge split": (len(training_positive), len(validation_positive), len(test_positive)),
       "validation AUC/AP": (round(validation_auc, 3), round(validation_ap, 3)),
       "test AUC/AP": (round(test_auc, 3), round(test_ap, 3))})
```

</details>

ROC-AUC 判断正例是否倾向于排在采样负例之前；average precision 对类别比例敏感。这个教学划分人为平衡正负例，因此其 AP 不能与包含数百万候选项的线上推荐系统直接比较。生产评估应复现 candidate generation、exposure、时间顺序、用户群体和 cold-start 条件。


### **图神经网络评估** {#graph-neural-network-evaluation}

图评估首先要确定独立单元，以及预测时允许使用哪些信息。单图上的随机节点划分测试的是**传导式**补全：测试节点在优化期间没有标签，但可以传递消息。归纳式节点测试要从训练中移除节点、邻域或整张图。边预测必须从消息传递图中删除留出的正边。图分类要按整图划分，通常还需要按 scaffold、entity group、source 或 time 分组，以防近重复泄漏。

指标取决于任务。节点和边分类根据不平衡程度及决策成本使用 macro/micro F1、balanced accuracy、AUROC 或 average precision。链接排序使用 MRR、Hits@$K$、Recall@$K$、NDCG 和与候选集合一致的 precision。图回归使用带有领域单位的 MAE/RMSE。当预测会驱动真实决策时，还应评估 calibration、subgroup performance、temporal degradation、latency、memory 和 sampling variance。

每个学习型 GNN 都应与非神经和 feature-only baseline 比较。图可能没有增加有效信号，或者标签仅靠 degree 就能预测。下面的代码在完全相同的测试节点上比较 majority rule、使用三个结构特征的 logistic regression，以及前面的 GCN；同时再次断言留出的链接边从未进入 encoder adjacency。

<details>
<summary><strong>Python：报告 baseline 并验证 split provenance</strong></summary>

```python
gcn_predictions = gcn_logits.argmax(1)
majority_class = int(torch.mode(labels[train_mask]).values)
majority_predictions = torch.full_like(labels[test_mask], majority_class)

feature_baseline = LogisticRegression(random_state=1018, max_iter=500)
feature_baseline.fit(node_features[train_mask].numpy(), labels[train_mask].numpy())
feature_predictions = feature_baseline.predict(node_features[test_mask].numpy())

evaluation_report = {
    "majority macro F1": f1_score(
        labels[test_mask].numpy(), majority_predictions.numpy(), average="macro"
    ),
    "feature-only macro F1": f1_score(
        labels[test_mask].numpy(), feature_predictions, average="macro"
    ),
    "GCN macro F1": f1_score(
        labels[test_mask].numpy(), gcn_predictions[test_mask].numpy(), average="macro"
    ),
    "clean link test ROC-AUC": test_auc,
    "clean link test AP": test_ap,
}

held_out_set = {tuple(sorted(edge)) for edge in validation_positive + test_positive}
training_set = {tuple(sorted(edge)) for edge in training_graph.edges()}
assert held_out_set.isdisjoint(training_set)
assert set(train_nodes).isdisjoint(test_nodes)
print({key: round(float(value), 3) for key, value in evaluation_report.items()})
```

</details>

这些数字有意不被包装成 leaderboard。单一社会网络中的节点相互相关，测试集只有 18 个节点，模型选择只看到 8 个 validation node，而且一个 seed 无法估计不确定性。可信研究应预先规定多个 split 或时间/scaffold 划分，公平地调优每个 baseline，在独立图或多个 seed 上报告 confidence interval，并按 degree 与 subgroup 检查性能。


### **章节对比与总结** {#chapter-comparison-summary}

图神经网络用关系邻域替代固定的空间或时间坐标。它的核心契约很简单：共享局部函数处理消息，顺序不变聚合处理邻居集合，任务 head 保持所需的节点等变性或图不变性。真正困难的是决定什么是节点、边、关系、时间戳、负例与独立评估单元。

| 方法或组件 | 邻域规则 | 主要优势 | 主要限制或诊断点 |
|---|---|---|---|
| Message-passing neural network | 学习消息，并用不变的 sum/mean/max 聚合 | 可统一处理节点、边和图特征 | 表达能力取决于聚合方式和可用图结构 |
| GCN | 固定的对称度归一化 | 简单、稳定，稀疏传播高效 | 传导式 identity feature 与深层 smoothing 会限制迁移和深度 |
| GraphSAGE | 采样特征聚合，并组合自身与邻域 | 具备归纳式函数和有界 fan-out | 仍存在采样方差与邻域爆炸 |
| GAT | 对 incident edge 学习局部 softmax | 依据内容调整邻居权重 | 逐边打分有成本；attention weight 不是因果解释 |
| Relation/temporal GNN | 按类型或时间调整消息 | 表示多种语义与事件顺序 | 参数增长、时间泄漏和采样复杂度 |
| Node readout | 每个节点状态产生一个预测 | 对实体分类或回归 | 相关节点的随机划分可能夸大泛化能力 |
| Edge readout | 对端点/关系进行 pairwise scoring | 链接与关系预测 | Negative sampling 和留出边泄漏会定义最终结果 |
| Graph readout | 不变 sum/mean/max 或学习型 pooling | 每张图产生一个预测 | Pooling 可能删除规模或局部子结构信息 |

Karate Club 实验把所有机制连接到一个可检查的图上：表示在节点置换下保持一致；GCN 对 degree 做归一化；GraphSAGE 采样可复用特征；GAT 在每个目标邻域内部归一化 attention；readout 与输出层级匹配；relation-aware 代码没有假装权重是时间戳；smoothing 与 bottleneck diagnostic 被明确分开；link evaluation 在传播前删除了留出边。

因此，实际决策顺序应当是：

1. 在选择层之前定义图语义与预测时间；
2. 选择节点、边或图级输出，并明确其对称性契约；
3. 建立 feature-only、topology-only 和简单 propagation baseline；
4. 根据规模与部署方式选择 GCN、GraphSAGE、GAT 或 typed/temporal message；
5. 检查 neighborhood growth、degree bias、oversmoothing、oversquashing 与 sampling variance；
6. 按真实部署单元划分，并审计每个 feature 和 edge 的泄漏。

GNN 性能永远不能只归因于神经层。图构造决定哪些信息可以传递，聚合决定哪些内容被保留，深度决定信息可能传播多远，而评估决定报告结果是否真的对应预期用途。
